In [1]:
import os
import shutil
import hashlib
import re
import requests
import json
import asyncio
import nest_asyncio
from pathlib import Path
from bs4 import BeautifulSoup
from playwright.sync_api import sync_playwright
from datetime import datetime

In [2]:
LEXES_API_URL = "https://polylex-admin.epfl.ch/api/v1/lexes?isAbrogated=0"
LANGUAGES = ["fr", "en"]
DATA_DIR = Path("data")
STATS_DIR = Path("stats")

In [3]:
# Fedlex

def resolve_redirect_sync(url):
    with sync_playwright() as p:
        browser = p.chromium.launch(headless=True)
        page = browser.new_page()
        page.goto(url, wait_until="domcontentloaded")
        page.wait_for_timeout(1500)
        redirect = page.url
        browser.close()
        return redirect

async def _resolve_redirect_async(url):
    return await asyncio.to_thread(resolve_redirect_sync, url)

def resolve_redirect(url):
    loop = asyncio.get_event_loop()
    return loop.run_until_complete(_resolve_redirect_async(url))

def get_fedlex_api_style_url(url):
    # FIXME : all manual tricks, a bit bad...
    api_url = url.replace("https://www.fedlex.admin.ch/", "https://fedlex.data.admin.ch/")
    api_url = api_url.replace("fedlex.data.admin.ch/eli/oc", "fedlex.data.admin.ch/eli/cc")
    url_end_exceptions = ["/fr", "/en", "/fr#a2"]
    for url_end in url_end_exceptions:
        if api_url.endswith(url_end):
            api_url = api_url.replace(url_end, "")
            break
    return api_url

def get_fedlex_pdf_url(url, lang):
    if lang == "en":
        lang_uri = "ENG"
    else:
        lang_uri = "FRA"
    # TODO : comprendre la requete -> base sur https://fedlex.data.admin.ch/fr-CH/sparql?id=101 puis ChatGPT
    sparql_query = f"""
PREFIX jolux: <http://data.legilux.public.lu/resource/ontology/jolux#>
SELECT ?publicationDate ?dateApplicability ?fileUrl WHERE {{
  ?work jolux:isMemberOf <{url}> ;
        jolux:dateApplicability ?dateApplicability ;
        jolux:isRealizedBy ?expr .
  OPTIONAL {{ ?work jolux:publicationDate ?publicationDate }}
  ?expr jolux:language <http://publications.europa.eu/resource/authority/language/{lang_uri}> ;
        jolux:isEmbodiedBy ?manif .
  ?manif jolux:userFormat <https://fedlex.data.admin.ch/vocabulary/user-format/pdf-a> ;
        jolux:isExemplifiedBy ?fileUrl .
}}
ORDER BY DESC(?publicationDate) DESC(?dateApplicability)
LIMIT 1
""".strip()
    endpoint = "https://fedlex.data.admin.ch/sparqlendpoint"
    r = requests.get(endpoint, params={"query": sparql_query, "format": "application/sparql-results+json"})
    r.raise_for_status()
    data = r.json()
    bindings = data["results"]["bindings"]
    if not bindings:
        print(f"No SPARQL results for {url}")
        return ""
    return bindings[0]["fileUrl"]["value"]

def get_fedlex_pdf_from_sparql(url, lang):
    redirected_url = resolve_redirect(url)
    sparql_url = get_fedlex_api_style_url(redirected_url)
    return get_fedlex_pdf_url(sparql_url, lang)

In [4]:
# retrieve metadata from Polylex API and Fedlex

def get_urls_from_html(text_in_html):
    soup = BeautifulSoup(text_in_html, "html.parser")
    urls = []
    for a in soup.find_all("a"):
        href = a.get("href", "").strip()
        if "mailto:" not in href:
            urls.append(href)
    return urls

def transform_html_in_text(text_with_html):
    soup = BeautifulSoup(text_with_html, "html.parser")
    for a in soup.find_all("a"):
        href = a.get("href", "").strip()
        label = a.get_text()
        a.replace_with(f"{label} ({href})" if href else label)
    for br in soup.find_all("br"):
        br.replace_with("\n")
    text = soup.get_text()
    text = text.replace("\xa0", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def transform_url(url, lang):
    transformed_url, source, content_format = "", "", ""
    if "inside.epfl.ch" in url:
        print(f"Can not load {url} (restricted access to EPFL members)")
        return transformed_url, source, content_format # empty value
    if "www.admin.ch" in url or "fedlex.admin.ch" in url:
        source = "fedlex"
        content_format = "pdf"
        if url.endswith(".pdf"):
            transformed_url = url
        else:
            if url == "http://www.admin.ch/ch/f/rs/22.html" or url == "https://www.admin.ch/opc/fr/classified-compilation/83.html":
                print(f"This page from Fedlex is not handled: {url}")
            else:
                transformed_url = get_fedlex_pdf_from_sparql(url, lang)
        return transformed_url, source, content_format
    if url.endswith(".pdf") or url.endswith(".docx"):
        transformed_url = url
        source = "others"
        content_format = "pdf" if url.endswith(".pdf") else "docx"
        return transformed_url, source, content_format
    epfl_redirect_urls_pattern = re.compile(r'^https://.*\.epfl\.ch$')
    epfl_websites_pattern = re.compile(r'^https://www\.epfl\.ch/(about|campus|education)/')
    epfl_apps_pattern = re.compile(r'(sac|isa)\.epfl\.ch')
    if url.endswith(".html") or epfl_redirect_urls_pattern.search(url) or epfl_websites_pattern or epfl_apps_pattern.search(url):
        # TODO : si site alors message d'avertissement et rien ou charger dans une cle fake tous les elements non charges ?
        print(f"{url} not loaded (website)")
        return transformed_url, source, content_format # empty value
    # TODO : a gerer dans les logs
    print(f"This is an exception and has to be handled: {url}")
    return transformed_url, source, content_format

def make_doc_id(key):
    return hashlib.sha256(key.encode("utf-8")).hexdigest()[:32]

def upsert_doc(data, key, cat, source, content_format, ref):
    if key not in data:
        data[key] = {
            "doc_id": make_doc_id(key),
            "cat": cat,
            "source": source,
            "content_format": content_format,
            "refs": []
        }
    data[key]["refs"].append(ref)

nest_asyncio.apply()

response = requests.get(LEXES_API_URL)
if response.status_code != 200:
    # TODO : a gerer dans les logs
    raise Exception(f"Unexpected status code while fetching : {response.status_code}")

data = {}

for lex in response.json():
    lex_id = lex.get('_id')
    lex_type = lex.get("type")
    lex_number = lex.get("number")

    for lang in LANGUAGES:
        lang = lang.capitalize()
        lex_url = lex.get(f"url{lang}")
        lex_description = lex.get(f"description{lang}")
        lex_summary = lex.get(f"title{lang}") + "\n" + transform_html_in_text(lex_description)
        lex_appendix_urls = get_urls_from_html(lex_description)
        ref = {
                "lex_id": lex_id,
                "lex_type": lex_type,
                "lex_number": lex_number,
                "lex_lang": lang
        }

        upsert_doc(data, lex_summary, "summary", "polylex", "txt", ref)

        transformed_lex_url, lex_source, lex_format = transform_url(lex_url, lang)
        if transformed_lex_url != "" and lex_source != "" and lex_format != "":
            upsert_doc(data, transformed_lex_url, "lex", lex_source, lex_format, ref)

        for lex_appendix_url in lex_appendix_urls:
            transformed_lex_appendix_url, lex_appendix_source, lex_appendix_format = transform_url(lex_appendix_url, lang)
            if transformed_lex_appendix_url != "" and lex_appendix_source != "" and lex_appendix_format != "":
                upsert_doc(data, transformed_lex_appendix_url, "appendix", lex_appendix_source, lex_appendix_format, ref)

https://www.epfl.ch/campus/services/communication/identite-visuelle/logo not loaded (website)
https://mediacom.epfl.ch not loaded (website)
https://www.epfl.ch/about/vice-presidencies/fr/vice-presidence-academique-vpa/textes-de-reference-vpa/ not loaded (website)
https://www.epfl.ch/about/news-and-media not loaded (website)
https://www.epfl.ch/campus/services/en/communication-en/visual-identity/logotype not loaded (website)
https://www.epfl.ch/about/vice-presidencies/vice-presidency-for-academic-affairs-vpa/reference-texts-vpa/ not loaded (website)
https://www.epfl.ch/about/vice-presidencies/fr/vice-presidence-academique-vpa/textes-de-reference-vpa/ not loaded (website)
https://www.epfl.ch/about/vice-presidencies/vice-presidency-for-academic-affairs-vpa/reference-texts-vpa/ not loaded (website)
https://citation.epfl.ch not loaded (website)
https://www.efv.admin.ch/efv/fr/home/themen/finanzpolitik_grundlagen/risiko_versicherungspolitik.html not loaded (website)
https://www.efv.admin.ch/

In [5]:
# load actual state of metadata in json file

os.makedirs(STATS_DIR, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
metadata_filename = os.path.join(STATS_DIR, f"{timestamp}_lexes_metadata.json")
stats_filename = os.path.join(STATS_DIR, f"{timestamp}_lexes_stats.csv")

with open(metadata_filename, "w", encoding="utf-8") as f:
    json.dump(data, f, indent=4, ensure_ascii=False)

In [6]:
# filter metadata to keep only content we want to (based on some rules)

# TODO : supprimer "lex_number": "5.1.0.3" car 448 pages et "lex_number": "1.1.11" car 92 pages (en tout 3 documents car 1 pas traduit)

In [7]:
# load all documents

def write_txt(filename, dir, content):
    with open(os.path.join(dir, filename), "w", encoding="utf-8") as f:
        f.write(content)

def download_file(url, dir, filename):
    headers = {
        "User-Agent": "Mozilla/5.0 (X11; Ubuntu; Linux x86_64; rv:147.0) Gecko/20100101 Firefox/147.0"
    }
    response = requests.get(url, headers=headers)
    if response.status_code == 200:
        with open(os.path.join(dir, filename), "wb") as f:
            f.write(response.content)
    else:
        print(f"Error: the content for {url} can not be fetched (status: {response.status_code})")

shutil.rmtree(Path(DATA_DIR), ignore_errors=True)
os.makedirs(DATA_DIR, exist_ok=True)

for content, metadata in data.items():
    doc_id = metadata.get("doc_id")
    content_format = metadata.get("content_format")
    if content_format == "txt":
        filename = f"{doc_id}.txt"
        write_txt(filename, DATA_DIR, content)
    elif content_format == "docx":
        filename = f"{doc_id}.docx"
        download_file(content, DATA_DIR, filename)
    elif content_format == "pdf":
        filename = f"{doc_id}.pdf"
        download_file(content, DATA_DIR, filename)
    else:
        print(f"File format not yet supported: '{content}'")

Error: the content for https://www.epfl.ch/about/overview/wp-content/uploads/2019/09/5.1.1_r_financier_fr.pdf can not be fetched (status: 404)
